# Track B: BRCA1 Mutation Walk -- Evo 2 (7B) Embeddings

**Purpose**: Embed the BRCA1 single-point mutation walk using Evo 2 (7B)
and cache the results for cross-model analysis.

| Model | Architecture | Params | Embedding |
|-------|-------------|--------|-----------|
| **Evo 2** (7B) | StripedHyena (Hybrid SSM + Attention) | 7B | Layer `blocks.28.mlp.l3` mean-pool |

## Setup
1. Upload `evaluation_harness.py` to `/content/`
2. Change Runtime to **A100 GPU** (7B model needs ~28 GB VRAM)
3. Run cells in order

## Dependency Notes
- Pins torch to 2.7.1+cu128 (Arc Institute recommended)
- Builds flash-attn 2.8.0.post2 from source (~10-15 min)
- Loads via `Evo2('evo2_7b')` (Vortex, not HuggingFace)
- Extracts embeddings from `blocks.28.mlp.l3` with `return_embeddings=True`

Results are cached to `./results/interpolation_track_b/cache/` for the analysis notebook.

In [ ]:
# Cell: Install Dependencies
#
# Evo 2 dependency strategy (matching MINE_03_Evo2 pattern):
#   1. Install core deps
#   2. Pin torch to 2.7.1+cu128 (Arc Institute recommended)
#   3. Build flash-attn from source (~10-15 min on A100)
#   4. Install evo2 package

print('Installing core dependencies...')
!pip install -q matplotlib seaborn pandas scipy biopython numpy scikit-learn
!pip install -q ninja

# Pin torch to 2.7.1+cu128 (Arc Institute recommended for evo2)
print('Pinning torch to 2.7.1+cu128...')
!pip install -q torch==2.7.1 --index-url https://download.pytorch.org/whl/cu128

import torch
print(f'torch {torch.__version__} | CUDA {torch.version.cuda}')

# Build flash-attn from source (~10-15 min on A100)
print('Building flash-attn 2.8.0.post2 from source (~10-15 min)...')
!pip install flash-attn==2.8.0.post2 --no-build-isolation

!pip install -q evo2

import torch
print(f'\ntorch {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    mem_gb = torch.cuda.get_device_properties(0).total_mem / 1e9
    print(f'GPU memory: {mem_gb:.1f} GB')

from evo2 import Evo2
print('\nEvo2 package loaded successfully')
print('Done!')

In [ ]:
# Cell: Configuration

import os
import sys
import numpy as np
import torch

sys.path.insert(0, '.')

# --- Output paths ---
OUTPUT_BASE = './results/interpolation_track_b/'
RESULTS_DIR = OUTPUT_BASE + 'results'
CACHE_DIR   = OUTPUT_BASE + 'cache'

os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(CACHE_DIR, exist_ok=True)

# --- BRCA1 ---
BRCA1_REGION_LEN = 2000   # 2kb core region for mutation walk
BRCA1_FLANKING   = 16_384 # Total region to fetch (with flanking context)
N_EXTRA_SNPS     = 120
SEED             = 320

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
print(f'Output: {OUTPUT_BASE}')

# --- Evo 2 ---
EVO2_CHECKPOINT = 'evo2_7b'  # Vortex checkpoint name (NOT HuggingFace repo)
EVO2_EMBEDDING_LAYER = 'blocks.28.mlp.l3'
EVO2_CONTEXT = 8192
print(f'Evo 2 checkpoint: {EVO2_CHECKPOINT}')
print(f'Evo 2 embedding layer: {EVO2_EMBEDDING_LAYER}')
print(f'Evo 2 context: {EVO2_CONTEXT}')

In [ ]:
# Cell: Download BRCA1 Region & Define Mutation Walk Endpoints

import urllib.request
import json as _json


def fetch_brca1_region(total_len=BRCA1_FLANKING, core_len=BRCA1_REGION_LEN):
    '''Fetch BRCA1 region from UCSC (GRCh38/hg38).

    Downloads total_len bp centered on the C61G variant position.
    The central core_len bp is the "mutation zone" where SNPs are introduced.
    '''
    center = 43_104_121  # BRCA1 C61G position (GRCh38)
    start = center - total_len // 2
    end = start + total_len
    core_start = (total_len - core_len) // 2
    core_end = core_start + core_len

    url = (f'https://api.genome.ucsc.edu/getData/sequence'
           f'?genome=hg38;chrom=chr17;start={start};end={end}')

    print(f'Fetching BRCA1 region chr17:{start:,}-{end:,} ({total_len:,} bp)...')
    try:
        req = urllib.request.Request(url)
        req.add_header('User-Agent', 'GeometricTax-Experiment/1.0')
        with urllib.request.urlopen(req, timeout=30) as response:
            data = _json.loads(response.read().decode())
            seq = data.get('dna', '').upper()
            if len(seq) >= total_len * 0.9:
                seq = seq[:total_len]
                rng = np.random.default_rng(SEED)
                seq = ''.join(
                    c if c in 'ACGT' else rng.choice(list('ACGT'))
                    for c in seq
                )
                print(f'  Downloaded {len(seq)} bp from UCSC')
                print(f'  Core region: positions [{core_start}:{core_end}] ({core_len} bp)')
                return seq, core_start, core_end
    except Exception as e:
        print(f'  UCSC download failed: {e}')

    # Fallback: synthetic
    print('  Generating synthetic BRCA1-like sequence as fallback...')
    rng = np.random.default_rng(SEED + 17)
    bases = rng.choice(list('ACGT'), size=total_len, p=[0.29, 0.21, 0.21, 0.29])
    seq = ''.join(bases)
    print(f'  Generated {len(seq)} bp synthetic BRCA1-like sequence')
    return seq, core_start, core_end


def create_pathogenic_variant(full_seq, core_start, core_end,
                              n_extra_snps=N_EXTRA_SNPS, seed=SEED):
    '''Create a pathogenic variant by mutating only the core region.'''
    rng = np.random.default_rng(seed)
    bases = list(full_seq)
    core_center = (core_start + core_end) // 2
    mutated_positions = set()

    # 1. Pathogenic mutation at core center
    alt_base = 'G' if bases[core_center] != 'G' else 'C'
    print(f'  Pathogenic mutation at position {core_center}: {bases[core_center]} -> {alt_base}')
    bases[core_center] = alt_base
    mutated_positions.add(core_center)

    # 2. Additional random SNPs within core region only
    available = [i for i in range(core_start, core_end) if i not in mutated_positions]
    snp_positions = rng.choice(available, size=min(n_extra_snps, len(available)),
                                replace=False)
    snp_positions.sort()

    for pos in snp_positions:
        original = bases[pos]
        alternatives = [b for b in 'ACGT' if b != original]
        bases[pos] = rng.choice(alternatives)
        mutated_positions.add(pos)

    mutant = ''.join(bases)
    hamming = sum(a != b for a, b in zip(full_seq, mutant))
    print(f'  Total mutations: {hamming} (all within core [{core_start}:{core_end}])')
    return mutant, sorted(mutated_positions)


# Generate endpoints
full_wt_seq, CORE_START, CORE_END = fetch_brca1_region()
full_mut_seq, mutation_positions = create_pathogenic_variant(
    full_wt_seq, CORE_START, CORE_END)

print(f'\nFull sequence length: {len(full_wt_seq)}')
print(f'Core region: [{CORE_START}:{CORE_END}] ({CORE_END - CORE_START} bp)')
print(f'Hamming distance: {sum(a != b for a, b in zip(full_wt_seq, full_mut_seq))}')

wildtype_seq = full_wt_seq
mutant_seq = full_mut_seq

In [ ]:
# Cell: Generate Single-Point Mutation Walk

def single_point_mutation_walk(seq_start, seq_end, seed=SEED):
    """Create a mutation walk from seq_start to seq_end, one base at a time."""
    assert len(seq_start) == len(seq_end), 'Sequences must be same length'

    diff_positions = [i for i in range(len(seq_start))
                      if seq_start[i] != seq_end[i]]

    rng = np.random.default_rng(seed)
    rng.shuffle(diff_positions)

    current = list(seq_start)
    walk = [''.join(current)]
    step_positions = []

    for pos in diff_positions:
        current[pos] = seq_end[pos]
        walk.append(''.join(current))
        step_positions.append(pos)

    assert walk[-1] == seq_end, 'Walk did not reach target'
    return walk, step_positions


mutation_walk, walk_positions = single_point_mutation_walk(wildtype_seq, mutant_seq)

n_steps = len(mutation_walk)
print(f'Mutation walk: {n_steps} steps (start + {n_steps - 1} mutations)')

center_pos = (CORE_START + CORE_END) // 2
pathogenic_step = None
for i, pos in enumerate(walk_positions):
    if pos == center_pos:
        pathogenic_step = i + 1
        break
print(f'Pathogenic C61G mutation occurs at step {pathogenic_step}/{n_steps - 1}')

walk_cache = f'{CACHE_DIR}/brca1_mutation_walk.npz'
np.savez(walk_cache, walk=mutation_walk, positions=walk_positions,
         pathogenic_step=pathogenic_step)
print(f'Walk cached to {walk_cache}')

In [ ]:
# Cell: Load Evo 2 (7B) via Vortex
#
# Uses the same loading pattern as MINE_03_Evo2.ipynb:
#   - Evo2('evo2_7b') via Vortex (not HuggingFace AutoModel)
#   - evo2_model.tokenizer.tokenize() for tokenization
#   - return_embeddings=True with layer_names for hidden state extraction
#   - Mean-pool across positions for the walk embedding

import torch
import gc
from evo2 import Evo2


def cleanup_gpu():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.synchronize()


def load_evo2(checkpoint_name=EVO2_CHECKPOINT):
    """Load Evo 2 (7B) via Vortex and return embedding function.

    Uses the native evo2 package interface (same as MINE notebook).
    Embeddings are extracted from an intermediate layer and mean-pooled.
    """
    print(f'Loading Evo 2 ({checkpoint_name}) via Vortex...')
    device = 'cuda:0' if torch.cuda.is_available() else 'cpu'

    evo2_model = Evo2(checkpoint_name)
    n_params = sum(p.numel() for p in evo2_model.model.parameters()) / 1e6
    layer_name = EVO2_EMBEDDING_LAYER

    if torch.cuda.is_available():
        mem = torch.cuda.memory_allocated() / 1e9
        print(f'  Params: {n_params:.1f}M | GPU mem: {mem:.2f} GB')
    print(f'  Embedding layer: {layer_name}')

    @torch.no_grad()
    def embed(sequences):
        """Embed sequences by mean-pooling layer activations."""
        embeddings = []
        n_total = len(sequences)
        for idx, seq in enumerate(sequences):
            if (idx + 1) % 5 == 0 or idx == 0 or idx == n_total - 1:
                print(f'  Sequence {idx+1}/{n_total}', end='\r')

            # Truncate to context length
            seq_input = seq[:EVO2_CONTEXT]

            # Tokenize using evo2's native tokenizer
            input_ids = torch.tensor(
                evo2_model.tokenizer.tokenize(seq_input), dtype=torch.int,
            ).unsqueeze(0).to(device)

            # Forward pass with embedding extraction
            _, layer_embs = evo2_model(
                input_ids, return_embeddings=True, layer_names=[layer_name],
            )

            emb = layer_embs[layer_name]  # (1, seq_len, d_model)
            pooled = emb.mean(dim=1).squeeze(0)  # (d_model,)
            embeddings.append(pooled.cpu().float().numpy())

        print()
        return np.array(embeddings, dtype=np.float32)

    print(f'Evo 2 ready ({n_params:.0f}M params, layer={layer_name})')
    return embed, evo2_model, n_params


evo2_embed, evo2_model, evo2_params = load_evo2()

In [ ]:
# Cell: Embed Mutation Walk with Evo 2
#
# NOTE: With 7B parameters, each sequence takes a few seconds on A100.
# Total wall time for ~120 steps: approximately 5-15 min.

evo2_cache = f'{CACHE_DIR}/evo2_brca1_walk.npy'

if os.path.exists(evo2_cache):
    print('Loading cached Evo 2 embeddings...')
    evo2_embeddings = np.load(evo2_cache)
else:
    # Truncate/pad walk sequences to Evo 2 context length
    evo2_walk = []
    for seq in mutation_walk:
        if len(seq) < EVO2_CONTEXT:
            evo2_walk.append(seq + 'N' * (EVO2_CONTEXT - len(seq)))
        else:
            evo2_walk.append(seq[:EVO2_CONTEXT])

    print(f'Embedding {len(evo2_walk)} walk steps with Evo 2 (7B)...')
    print(f'(Context: {EVO2_CONTEXT} bp, layer: {EVO2_EMBEDDING_LAYER})')
    evo2_embeddings = evo2_embed(evo2_walk)
    # Clean any NaN/Inf from fp16 artifacts
    evo2_embeddings = np.nan_to_num(evo2_embeddings, nan=0.0, posinf=1e6, neginf=-1e6)
    np.save(evo2_cache, evo2_embeddings)
    print(f'Cached to {evo2_cache}')

print(f'Evo 2 embeddings: {evo2_embeddings.shape}')

# Free GPU memory
del evo2_model
cleanup_gpu()
print('Evo 2 model freed from GPU')

In [ ]:
# Cell: Quick Sanity Check

from scipy.spatial.distance import cosine

print('Sanity check: cosine distance from wildtype')
for step_idx in [0, 10, 50, len(evo2_embeddings) - 1]:
    if step_idx < len(evo2_embeddings):
        d = cosine(evo2_embeddings[0], evo2_embeddings[step_idx])
        print(f'  Step {step_idx:4d}: cosine dist = {d:.6f}')

# Consecutive step distances
dists = [cosine(evo2_embeddings[i], evo2_embeddings[i + 1])
         for i in range(len(evo2_embeddings) - 1)]
dists = np.array(dists)
print(f'\nConsecutive step cosine distances:')
print(f'  mean={dists.mean():.6f}, std={dists.std():.6f}, max={dists.max():.6f}')
print(f'  smoothness (mean/max): {dists.mean() / (dists.max() + 1e-8):.4f}')
print(f'\nEvo 2 embeddings cached and ready for analysis notebook.')